# Initial Value Problem for ODE: single-step methods

Let’s look back at what we have achieved so far:

- We can use finite differences to numerically approximate derivatives
- We can derive these finite difference approximations from a Taylor series expansion, giving us an idea about the error of the approximation.

Next, let us see how we can apply these concept in order to numerically integrate (that is, solve) ODEs.

The solution process of an ODE is performed by steps. In a single-step approach, the solution of the following step depends only on the current one. In a multiple step method, the solution of the next step is calculated from several steps. A multiple step approach is more accurate, you can think of a similar construct as the higher-order derivatives treated previously. 

For first order ODEs with the general form: 

$$
\frac{dy}{dx} = f(x,y)
$$

**One initial condition is needed to find only one solution!** Without it, there would be an infinite number of solutions possible. The initial condition is:

$$
y(x_0)=y_0
$$

## Forward (Explicit) Euler

The simplest initial value numerical integration is the Forward Euler.
The Forward Euler method can be derived from the backwards finite difference approximation of the derivative.
Recall the backwards finite difference formula:

$$
y'(x_i)=\frac{y(x_{i+1})-y(x_i)}{\Delta x} + \mathcal{O}(\Delta x)
$$

where 
$$
x_{i+1}=x_i+\Delta x 
$$

For simplicity, we can also define $y_{i} = y(x_{i})$ and write:

$$
y'(x_i)=\frac{y_{i+1}-y_i}{\Delta x} + \mathcal{O}(\Delta x)
$$

Now we can solve this expression for the state at the new time step, $y_{i+1}$:

$$
y_{i+1} = y_i + \Delta x y'(x_i) + \mathcal{O}(\Delta x)^2
$$

Ignoring the error terms, the Forward Euler formula becomes:

$$
y_{i+1} = y_i + \Delta x f(x_i, y_i)
$$

where we substituted $y'(x_i) = f(x_i, y_i)$.
Note that the derivative of $y$ (the slope) is evaluated at the current time step $x_i$.
This ma

which is an explicit method. Although simple, it contains the basic characteristics as more advanced and accurate methods. It looks as:

$$
x_{i+1}=x_i+\Delta x 
$$

$$
y_{i+1}=y_i+\Delta x \cdot slope \rvert_i
$$

where $\Delta x$ is the step size and the slope is a constant that approximates the rate of change of $y$ with respect to $x$ (a.k.a. the derivative) in the interval $x_i$ to $x_{i+1}$. The solution starts at $i=0$ given by the initial condition, then $i$ is increased to 1 where the values are calculated using the previous equations. This loop continues until the points cover the desired domain. **The computation of the slope is the key difference between single step methods. For forward euler, the slope is computed with the values at the current step!** 


```{figure} https://files.mude.citg.tudelft.nl/explicit_euler.png
:name: explicit_euler

Illustrating the forward Euler method
```



### Forward Euler Example

Consider the following equation: 

$$
\frac{dy}{dx}= y'=-y
$$
We want to know the solution in the domain $[a=0,b=30]$. The **initial condition** is $y(x_0)=1$ and the step size is $\Delta x = 0.2$.

The discretization of the differential equation transforms the problem into an algebraic one. Following the formula of explicit Euler for the discretization yields: 

$$
y_{i+1}= y_i + \Delta x *y'_i
$$

In the following code, Forward Euler is implemented.





In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

dx = .2
x = np.arange(0,30+dx,dx)
y = np.zeros(len(x))

##-----------------------------
##Forward Euler Implementation
##-----------------------------
alpha= 1
y[0] = 1
for i in range(len(x)-1): 
    y[i+1] = y[i] + dx*( -alpha*y[i])
##------------------------------

y_exact = np.exp(-alpha*x)
plt.plot(x,y_exact)
plt.plot(x,y)
plt.legend(['exact solution','Forward Euler solution'])


:::{card} Exercise 

The **Forward Euler** is a Finite Difference Method where it uses a grid-based approach and computes the solution at the nodes.

Can you show how the Forward Euler gets derived from the Finite Difference method

hint: Use the forward difference that we derived earlier in the Taylor Series Expansion [chapter](./4-taylor-series-expansion.ipynb) .

```{admonition} Solution
:class: tip, dropdown

In the Taylor expansion chapter we derived the forward difference for a first derivative: 

$$
f'(x_i)=\frac{f(x_{i+1})-f(x_i)}{\Delta x} + \mathcal{O}(\Delta x)
$$

$$
\Delta xf'(x_i)= f(x_{i+1})-f(x_i)+ \mathcal{O}(\Delta x)^2
$$

$$
f(x_{i+1}) = f(x_i) + \Delta x f'(x_i) + \mathcal{O}(\Delta x)^2
$$

```

:::

:::{card} Exercise

Use the forward Euler to approximate the solution to the initial-value problem.

$$
\frac{dy}{dt}= y^2, \hspace{5mm}, 0\leq t \leq 1
$$

$$
y(t_0)= 1
$$

with $\Delta t=0.5$

```{admonition} Solution
:class: tip, dropdown

$t_0= 0, t_1= 0.5, t_2= 1$

Use the forward euler:

$$
y_{i+1}=y_i+\Delta x \cdot y'_{i}
$$

for i =0

$$
y_{1}=y_0+\Delta x \cdot y'_{0} = 1+ 0.5 \cdot 1 = 1.5
$$

for i = 1

$$
y_{2}=y_1+\Delta x \cdot y'_{1} = 1.5+ 0.5 \cdot 1.5^2 = 2.625
$$

The approximated solution is 2.625

```
:::


## Error analysis

There are two types of errors: round-off and truncation. The **round-off** error is due to the computer's limitation to represent a floating number (decimal). To illustrate it consider the following: the difference between 1 and 0.9 is 0.1. If you subtract 0.1 from itself, you should obtain 0, even if this subtraction is repeated ten thousand times. However, in practice, an extremely small error accumulates over each repitition, so the result will not be zero. The following code demonstrates this behavior, using floating-point numbers with varying precision.  

In [ ]:
def accumulated_error(a,b,solution,iterations):
    error = solution - np.abs(a-b) 
    error_accumulated = 0
    for i in range(iterations):
        error_accumulated = error_accumulated + error

    return error_accumulated

print('Error using 16 bits of memory = ',accumulated_error(np.float16(1.),np.float16(0.9),np.float16(0.1),10000))
print('Error using 32 bits of memory = ',accumulated_error(np.float32(1.),np.float32(0.9),np.float32(0.1),10000))
print('Error using 64 bits of memory = ',accumulated_error(np.float64(1.),np.float64(0.9),np.float64(0.1),10000))


As you can see, this simple operation can give discernible errors. Imagine a computation spanning 100 years with a time step of seconds and more complex operations: the round-off error will be present! As you can see, this can be reduced by increasing the precision or the number of digits used to represent numbers but be careful, this is not free as the memory the computer uses increases as well as the computation time. 

The truncation error is related to the method chosen for the slopes approximation. This can be obtained using our reliable TSE as this gives the exact solution. Therefore **the truncation error per step** is 

$$
\text{Truncation error }= y^{TSE}(x_{i+1}) - y^{\text{Forward Euler}}(x_{i+1}) 
$$

$$
\text{Truncation error }= y(x_i)+\Delta x y'_i  +\frac{(\Delta x)^2}{2!} y''_i + ... - y(x_i)-\Delta x y'_i  \approx \mathcal{O}(\Delta x^2)
$$

The **total truncation error** accounts for the number of steps as: 

$$
\text{Total truncation error } \approx  \sum_{i=0}^{n-1} \frac{(\Delta x)^2}{2!} y''_i \approx \frac{(\Delta x)^2}{2!} \frac{b-a}{\Delta x} \bar{y}'' \approx \mathcal{O}(\Delta x) 
$$

In honor to this first order total truncation error, Forward Euler is referred to as a first-order method.


## Stability

The error that is introduced in each step of the numerical solution ideally does not to increase as the solution advances. Under a well posed and proper solution, the error is expected to reduce with smaller steps. In some cases, the error increases without bound as the solution advances from the initial condition (even with smaller steps): **the solution becomes unstable**. The stability depends on the numerical method, the step size and the behavior of the differential equation. Therefore, the stability conditions will differ when applying the same numerical method to different equations. 

Let's consider the **stability for Forward Euler** applied to a more general form of the problem above.  

$$
\frac{dy}{dx} = y'=-\alpha y
$$

With initial condition $y(0)=1$ and $\alpha>0$, the exact solution is:

$$
y^{\text{exact}}(x_{i})=y_0 e^{-\alpha x_{i}}
$$

The Forward Euler equivalent is 

$$
y_{i+1}=y_{i}-\alpha y_i \Delta x=y_i(1-\alpha \Delta x)
$$

Following the initial steps of the numerical solution a pattern arises:

$$
y_{1}=y_0(1-\alpha \Delta x)
$$

$$
y_{2}=y_1(1-\alpha \Delta x) =y_0(1-\alpha \Delta x)^2 
$$

$$
y_{3}=y_2(1-\alpha \Delta x) =y_0(1-\alpha \Delta x)^3 
$$

$$
y_{n}=y_0(1-\alpha \Delta x)^n 
$$


Comparing this last equation with the exact solution, it can be seen that the term $(1-\alpha \Delta x)^n$ in the numerical solution is approximating the term $e^{-\alpha \Delta x_i}$ in the exact solution. The latter tends to decay with larger values of $x_i$ and positive $\alpha$. To make sure that the term $(1-\alpha \Delta x)^n$ decays with larger $n$ values $1-\alpha \Delta x$ should be less than $|1|$, i.e.,

$$
|1-\alpha \Delta x| < 1 \implies 0 < \alpha \Delta x < 2
$$

```{admonition} Click here for a more detailed derivation
:class: tip, dropdown
$$
-1 <1-\alpha \Delta x < 1 
$$

Subtract 1 of both sides

$$
-2 <-\alpha \Delta x < 0
$$

Divide by -1

$$
0 <\alpha \Delta x < 2
$$

```

This is the stability criterion. If not complied, then the solution will be unstable. It is said that **Forward Euler is conditionally stable! This is true for every explicit numerical method.** 



:::{card} Exercise

Lets go back to the last example described in the code above.

$$
\frac{dy}{dx}= y'=-y
$$
We want to know the solution in the domain $[a=0,b=30]$. The **initial condition** is $y(x_0)=1$ and the step size is $\Delta x = 0.2$.

What is the step size at which the problem described using the Forward Euler becomes unstable? Feel free to use the interactive figure below and change dx to test what happens when the step size becomes larger.

Click `rocket` -->`Live Code` to interact with the plot below

:::

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets

def forward_euler(dx):
    # Define the x range and initialize y
    x = np.arange(0, 30 + dx, dx)
    y = np.zeros(len(x))
    
    # Forward Euler Implementation
    alpha = 1
    y[0] = 1
    for i in range(len(x) - 1):
        y[i + 1] = y[i] + dx * (-alpha * y[i])
    
    x_exact = np.arange(0, 30 + dx, 0.2)
    # Exact solution
    y_exact = np.exp(-alpha * x_exact)
    
    # Plot both solutions
    plt.figure(figsize=(8, 5))
    plt.plot(x_exact, y_exact, label='Exact solution', linestyle='--')
    plt.plot(x, y, label='Forward Euler solution', marker='o')
    plt.legend()
    plt.title(f"Forward Euler with dx = {dx}")
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.show()

# Create an interactive slider for dx
interact(forward_euler, dx=widgets.FloatSlider(value=0.2, min=0.01, max=3.0, step=0.01, description='dx'));


```{admonition} Solution
:class: tip, dropdown

You can see in the interactive figure above that when the stepsize becomes larger the forward euler solution becomes less accurate. You can also see that when the step size becomes bigger then 2 the Forward Euler solution starts to blow up. This is because of the stability criteria of the Forward Euler.

$$
0 <\alpha \Delta x < 2
$$

When $\alpha=1$, then $\Delta x < 2$  (Note that $\Delta x$ can not be negative and therefore will always be > 0 )

```

## Backward (Implicit) Euler



Just as Forward Euler is the simplest Explicit method, **Backward Euler method is the simplest implicit method**. They look very similar but the slope is computed at the next step! That's where the implicit term comes from: the slope depends on the unknown value.

$$
x_{i+1}=x_i+\Delta x 
$$
$$
y_{i+1}=y_i+\Delta x \cdot slope \rvert_{i+1}=y_i+\Delta x \cdot y'_{i+1}
$$

Let's consider the same problem as before: 

$$
\frac{dy}{dx}= y'=-\alpha y
$$
We want to know the solution in the domain $[a=0,b=30]$. The **initial condition** is $y(x_0)=1$ and the step size is $\Delta x = 0.2$.

The discretization of the differential equation transforms the problem into an algebraic one. Following the formula of implicit Euler, the discretization yields: 

$$
y_{i+1}= y_i - \Delta x \cdot \alpha y_{i+1}
$$

then

$$
y_{i+1}+ \Delta x \cdot \alpha y_{i+1} = y_i  
$$

finally

$$
y_{i+1}= y_i/(1+\Delta x \cdot \alpha)  
$$


In the following code, Implicit Euler is implemented.

In [ ]:
dx = 0.3
x = np.arange(0,30+dx,dx)
y = np.zeros(len(x))

##-----------------------------
##Backward Euler Implementation
##-----------------------------
alpha=1
y[0] = 1
for i in range(len(x)-1): 
    y[i+1] = y[i]/(1+dx*alpha)
##------------------------------

y_exact = np.exp(-alpha*x)
plt.plot(x,y_exact)
plt.plot(x,y)
plt.legend(['exact solution','Backward Euler solution'])

:::{card}
You can see that the result is similar to Forward Euler, except that the curve for Backward Euler is slightly above than the exact solution. This makes sense as the derivative is taken to be in front, so the slope is initially underestimated while the explicit solution initially overestimated the slope. 

The round-off errors and truncation error are similar. Backward Euler is also a first-order method. But what about the stability? 
:::

:::{card} Exercise:

Modify the step size in the Implicit Euler code, try to make the solution unstable. What do you notice?

:::

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets

def backward_euler(dx):
    # Define the x range and initialize y
    x = np.arange(0, 30 + dx, dx)
    y = np.zeros(len(x))
    
    ##-----------------------------
    ##Backward Euler Implementation
    ##-----------------------------
    alpha=1
    y[0] = 1
    for i in range(len(x)-1): 
        y[i+1] = y[i]/(1+dx*alpha)
    ##------------------------------
    
    x_exact = np.arange(0, 30 + dx, 0.2)
    # Exact solution
    y_exact = np.exp(-alpha * x_exact)
    
    # Plot both solutions
    plt.figure(figsize=(8, 5))
    plt.plot(x_exact, y_exact, label='Exact solution', linestyle='--')
    plt.plot(x, y, label='Backward Euler solution', marker='o')
    plt.legend()
    plt.title(f"Backward Euler with dx = {dx}")
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.show()

In [ ]:



# Create an interactive slider for dx
interact(backward_euler, dx=widgets.FloatSlider(value=0.2, min=0.01, max=5.0, step=0.01, description='dx'));




Let's understand what happens by following the same procedure as for the stability analysis of Explicit Euler.  

The exact solution is the same:

$$
y^{\text{exact}}(x_{i})=y_0 e^{-\alpha x_{i}}
$$


For the numerical solution, the numerical solution pattern is slightly different:

$$
y_{1}=y_0\left(\frac{1}{1+\alpha \Delta x}\right)
$$

$$
y_{2}=y_1\left(\frac{1}{1+\alpha \Delta x}\right) =y_0\left(\frac{1}{1+\alpha \Delta x}\right)^2
$$

$$
y_{3}=y_2\left(\frac{1}{1+\alpha \Delta x}\right) =y_0\left(\frac{1}{1+\alpha \Delta x}\right)^3
$$

$$
y_{n}=y_0\left(\frac{1}{1+\alpha \Delta x}\right)^n
$$

This last term to the power $n$ approximates the decaying exponential in the exact solution, just like in the Explicit Euler method. Here, the only condition that must be satisfied for stability is

$$\left|\frac{1}{1+\alpha \Delta x}\right| < 1$$

This can be expressed as:

$$ -1 <\frac{1}{1+\alpha \Delta x} < 1$$

Lets first look at the right side condition $\frac{1}{1+\alpha \Delta x} < 1$.

If we multiply by the term $1+\alpha \Delta x$, we get:

$$
1 < 1+\alpha \Delta x
$$

As $\alpha$ is positive and the step size must be larger than 0, the right side of the criterion is always true.

Lets now look at the left side condition $-1 <\frac{1}{1+\alpha \Delta x}$:

We again multiply by $1+\alpha \Delta x$:

$$
-1 - \alpha \Delta x < 1
$$

Again this left side criterion always holds.

 Hence, **an implicit Backward Euler scheme is unconditionally stable.** This is also the case for more advanced implicit schemes.

### Global errors

The global error is the sum of round-off and total truncation errors. The plot below displays the global errors for both the Forward and Backward Euler methods, alongside the reference line representing the slope of the truncation error $\mathcal{O}(\Delta x)$ (1 to 1). As shown, the global errors for both methods align with the expected slope, indicating that the errors decrease in accordance with the methods' first-order accuracy.

$$
\text{Global Error} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( y_i - y_{\text{exact},i} \right)^2}
$$



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def global_error(dx_values):
    
    errors_forward = []
    errors_backward = []
    
    for dx in dx_values:
        
        x = np.arange(0, 30 + dx, dx)
        y_forward = np.zeros(len(x))
        y_backward = np.zeros(len(x))
          
        alpha = 1
        y_forward[0] = 1
        for i in range(len(x) - 1):
            y_forward[i + 1] = y_forward[i] + dx * (-alpha * y_forward[i])
        
        y_backward[0] = 1
        for i in range(len(x) - 1): 
            y_backward[i + 1] = y_backward[i] / (1 + dx * alpha)
        
        y_exact = np.exp(-alpha * x)
        n= (30-0)/dx
        # Compute the global error (L2 norm) for both methods
        error_forward = np.sqrt((1/(n-1))*np.sum((y_forward - y_exact) ** 2))
        error_backward = np.sqrt((1/(n-1))*np.sum((y_backward - y_exact) ** 2))
        
        errors_forward.append(error_forward)
        errors_backward.append(error_backward)
    
    # Plot the global error vs. dx in a loglog plot for both methods
    plt.figure(figsize=(8, 5))
    plt.loglog(dx_values, errors_forward, label='Forward Euler Global error', marker='o')
    plt.loglog(dx_values, errors_backward, label='Backward Euler Global error', marker='s')
    plt.loglog(dx_values, [dx**1 for dx in dx_values], label=r'$\mathcal{O} (\Delta x)$', linestyle='--')
    
    plt.title(r"Global error comparison: Forward vs Backward Euler")
    plt.xlabel(r'$\Delta x$')
    plt.ylabel('Global error')
    plt.grid(True)
    plt.legend()
    plt.show()


dx_values = [0.00125, 0.0025, 0.005, 0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64]
# Call the function to plot the global error for both Forward and Backward Euler
global_error(dx_values)


## Nonlinear ODE 

In the previous example, the explicit and implicit implementation seem to only differ in the formulation. However, this changes when treating non-linear equations. Let's see an example to identify the "implicitness".

### An example: mixed bioreactor

Consider a bioreactor that is continuously stirred and where microorganisms degrade a chemical compound (see {numref}`bioreactor`).
A solution containing the compund at concentration $c_{\mathrm{in}}$ is fed into the reactor with flow rate $Q$. The same amount of water is removed from the reactor at the outflow.
The concentration inside the reactor is desginated by $c$, and the reactor volume by $V$.

```{image} https://files.mude.citg.tudelft.nl/mixed_reactor_bio.svg
:name: bioreactor
:alt: Conceptual scheme of a mixed bioreactor
:width: 100%
:figclass: margin

Conceptual model of a the mixed bioreactor
```

The reaction rate $r$ with which the bacteria degrade the compound depends non-linearly on the concentration:
at low concentrations the increase is linear, but when the concentration increases, the rate levels off and converges towards a constant maximum rate, $r_{\mathrm{max}}$.
This can be described by the so called Monod kinetics:

$$r = r_{\mathrm{max}}\frac{c}{c + K}$$

where the parameter $K$ indicates the concentration where the rate reaches half its maximum value.

Based on a mass balance, we can set up the following ODE to describe how the concentration in the reactor changes over time:

$$
\frac{dc}{dt} = \frac{Q}{V} \left(c_{\mathrm{in}}(t) - c\right) - r_{\mathrm{max}}\frac{c}{c + K} = f(c, t)
$$

The initial concentration in the reactor is $c(t=0) = c_0$.
To predict how the concentration in the reactor evolves over time, we need to solve the ODE.
Let's see how this looks like for an implicit and explicit Euler scheme.

The expression for the explicit Euler scheme is:

$$
    c_{i+1} = c_i + \Delta t f(c_i, t_i) = c_i + \Delta t \left[\frac{Q}{V} \left(c_{\mathrm{in}}(t_i) - c_i\right) - r_{\mathrm{max}}\frac{c_i}{c_i + K}\right]
$$

The algorithm to solve the explicit Forward Euler is exactly the same as before.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# set up time vector
dt = 0.2
t_end = 100
n_times = int(t_end / dt)
t = np.linspace(0, t_end, num=n_times)

# parameter values
c0 = 80.0
K = 50.0
c_in = 50.0
r_max = 5.0
Q = 0.1
V = 2.0

##-----------------------------
##Explicit Euler Implementation
##-----------------------------
c = [c0]
for i in range(n_times - 1):
    c_i = c[-1]
    dcdt = Q / V * (c_in - c_i) - r_max * c_i / (c_i + K)
    c.append(c_i + dt * dcdt)
##------------------------------

plt.plot(t, c)
plt.legend(["Forward Euler solution"])
plt.xlabel("time")
plt.ylabel("concentration")

Now, using the implicit Euler scheme, the expression becomes:

$$
    c_{i+1} = c_i + \Delta t f(c_{i+1}, t_{i+1}) = c_i + \Delta t \left[\frac{Q}{V} \left(c_{\mathrm{in}}(t_{i+1}) - c_{i+1}\right) - r_{\mathrm{max}}\frac{c_{i+1}}{c_{i+1} + K}\right]
$$

Even though this equation looks similar to the explicit Euler scheme at a first glance, it requires a different procedure because now the unknown $c_{i+1}$ appears on both sides of the equation.
Since the equation is non-linear, we cannot solve dircetly for $c_{i+1}$.
Instead, an iterative procedure is needed, for example, the widely used Newton-Rhapson method.

### Newton-Raphson Method

The Newton-Rapshon method is a numerical technique to find the roots of a function, that is, the locations where its value equals zero.
How does that help us to solve our implicit Euler expression for $c_{i+1}$?
Note that we can easily rewrite the equation such that the right-hand-side is zero, by moving all terms from the right to the left:

$$
    \underbrace{c_{i+1} - c_i - \Delta t \left[\frac{Q}{V} \left(c_{\mathrm{in}}(t_{i+1}) - c_{i+1}\right) - r_{\mathrm{max}}\frac{c_{i+1}}{c_{i+1} + K}\right]}_{q(c_{i+1})} = 0
$$

That is, the expression on the left is the function that we want to find the root of – let's call it $q(c_{i+1})$, and the root will equal the new concentration $c_{i+1}$ that we are looking for. So let's see how the Newton-Raphson method works.

We consider a function $g(z)$, and we are looking for the value $z$ where $g(z) = 0$.
The solution is graphically represented by the intersection of $g(z)$ with the $z$ axis (see the Figure below).
The key idea of the Newton-Raphson method is to linearize the root finding problem.
That is, we find the root of a tangent to the function $g(z)$ at a point $z_0$. Since the tangent is a simple line, it is easy to find its intersection with the $y$-axis.
This will not directly bring us to the correct solution for our original problem (finding the root of $g(z)$), but if we repeat the approach, it will iteratively bring us closer. This is illustrated by {numref}`newton-raphson`.

```{figure} https://files.mude.citg.tudelft.nl/newton-rhapson.svg
:name: newton-raphson
:width: 60%

Illustration of the Newton-Raphson method
```

Step by step, the procedure for the Newton-Raphson method is as follows:

1. We start with an initial guess $z_0$ and compute the tangent of the function $g(z)$ at this point.
   The tangent $t(z)$ has a slope of $g^\prime(z)$ and is described by $t(z) = g(z_0) + (z-z_0) \,g^\prime(z_0)$.
2. We follow the tangent to the point where it crosses the $z$-axis. Setting $t(z) = 0$, we can easily see that this is at
   $z_1 = z_0 - \frac{g(z_0)}{g^\prime(z_0)}$. Note how this point is already much closer to the root of $g(z)$.
3. Now, we repeat the procedure. That is, we compute the tangent at point $z_1$ and find its root to get to a new point $z_2$.
   Each new point $z_{j+1}$ can be obtained from the previous one as:
   $$z_{j+1} = z_j - \frac{g(z_j)}{g^\prime(z_j)}$$
4. We stop the process when the value of $g(z_j)$ becomes very small: $|g(z_j)<\epsilon|$. Here, $\epsilon$ is the tolerance e.g., $\epsilon=10^{-6}$ (see the red square in {numref}`newton-raphson`).



### Application to the implicit Euler scheme

We now apply this procedure to the expression we obtained from the implicit Euler scheme, to find the new concentration $c_{i+1}$.
Remember the function $q(c_{i+1})$ that we defined earlier:

$$
q(c_{i+1}) = c_{i+1} - c_i - \Delta t \left[\frac{Q}{V} \left(c_{\mathrm{in}}(t_{i+1}) - c_{i+1}\right) - r_{\mathrm{max}}\frac{c_{i+1}}{c_{i+1} + K}\right] = 0
$$

More generally, we can also write this for any ODE of the form $\frac{dc}{dt} = f(c, t)$ as:

$$
q(c_{i+1}) = c_{i+1} - c_i - \Delta t f(c_{i+1}, t_{i+1}) = 0
$$

Since $q(c_{i+1})$ corresponds to the function $g(z)$, we can iteratively find new values for $c_{i+1}^{j+1}$ with the following equation:

$$c_{i+1}^{j+1} = c_{i+1}^j - \frac{q(c_{i+1}^j)}{q^\prime(c_{i+1}^j)}$$

Note that the index $i$ refers to the time step of the implicit Euler method, whereas the index $j$ refers to the iteration of the Newton-Raphson scheme.
To compute the value $c_{i+1}^{j+1}$, we first need to evaluate $q^\prime$, the derivative of $q$ with respect to $c_{i+1}$.

$$q^\prime(c_{i+1}) = 1 - \Delta t f^\prime(c_{i+1}, t_{i+1})$$

where $f^\prime = \frac{\partial f(c, t)}{\partial c}$.

For the ODE describing the bioreactor, this is:

$$q^\prime(c_{i+1}) = 1 + \Delta t \left[\frac{Q}{V} + r_{\mathrm{max}} \frac{K}{(c_{i+1}+K)^2}\right]$$

The final expression for $c_{i+1}^{j+1}$ then becomes:

$$c_{i+1}^{j+1} = c_{i+1}^j - \frac{c_{i+1}^j - c_i - \Delta t f(c_{i+1}^j, t_{i+1})}{1 - \Delta t f^\prime(c_{i+1}^j, t_{i+1})}$$

And for the bioreactor:

$$c_{i+1}^{j+1} = c_{i+1}^j - \frac{c_{i+1}^j - c_i - \Delta t \left[\frac{Q}{V} \left(c_{\mathrm{in}}(t_{i+1}) - c_{i+1}^j\right) - r_{\mathrm{max}}\frac{c_{i+1}^j}{c_{i+1}^j + K}\right]}{1 + \Delta t \left[\frac{Q}{V} + r_{\mathrm{max}} \frac{K}{(c_{i+1}^j+K)^2}\right]}$$

This expression might look a bit intimidating when written out fully, but remember that when implementing the procedure in code, you can break it down into several steps. For example, define expressions (or functions) for $q$ and $q^\prime$ first and use them for defining $c_{i+1}^{j+1}$.
In practice, the implementation of the implicit Euler method with Newton-Raphson iterations in Python will require two nested loops: an outer loop for time stepping, and an inner loop for the Newton-Raphson iterations.


### Advantages and disadvantages of the Newton-Raphson method

The Newton-Raphson method

- converges quickly and can provide highly accurate approximations,
- can be used for both real and complex roots and
- can be extended to multi-dimensional problems (e.g., a system of coupled ODE).

But the method also has its limitations:

- It may not converge if the initial guess is far from the actual value or if the function is discontinuous near the value.
- The derivative of the function is needed, which may not always be known.
- For multi-dimensional problems, the Newton-Raphson method requires inverting the Jacobian, which is computationally expensive. The Jacobian is the matrix of partial derivaties for all equations with respect to all states.


% START-CREDIT
% source: numerical_modelling
```{attributiongrey} Attribution
:class: attribution
This chapter is written by Jaime Arriaga Garcia, Justin Pittman and Robert Lanzafame. {ref}`Find out more here <numerical_modelling_credit>`.
```
% END-CREDIT